In [1]:
# --- Package import ---

import pandas as pd
import folium
import numpy as np
from scipy.spatial.distance import cdist

import calliope

from ruamel.yaml import YAML

import geopandas as gpd
from shapely.geometry import Point, LineString, Polygon
from shapely.ops import snap
import io

import requests
import os

import xarray as xr

In [2]:
# --- Timing infrastructure ---

import time

# Global timing dictionary
execution_times = {}

class Timer:
    def __init__(self, name):
        self.name = name
    
    def __enter__(self):
        self.start = time.perf_counter()
        return self
    
    def __exit__(self, *args):
        self.end = time.perf_counter()
        elapsed = self.end - self.start
        execution_times[self.name] = elapsed
        print(f"✓ {self.name}: {elapsed:.3f}s")

In [3]:
# --- Select run mode: 'plot', 'export'

mode='plot'

In [4]:
# --- API calls to obtain geodata ---

with Timer("API calls to obtain geodata"):
    # Fetch building GeoJSON data for area 4011 from the TNO Warmteprofielgenerator API (may not be legal)
    geojson_url = "https://hlc-api.warmteprofielengenerator.nl/building_data/geojson/4011"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:145.0) Gecko/20100101 Firefox/145.0",
        "Accept": "application/json",
        "Referer": "https://www.warmteprofielengenerator.nl/",
        "Origin": "https://www.warmteprofielengenerator.nl"
    }

    response = requests.get(geojson_url, headers=headers)
    response.raise_for_status()
    data = response.json()

    # Extract properties from each feature
    if "features" in data and len(data["features"]) > 0:
        properties_list = [f.get("properties", {}) for f in data["features"]]
        residential_heat_demand = pd.DataFrame(properties_list)
    else:
        print("No features to export.")

    # Extract only the heat demand data from the InfluxDB/Grafana API
    url = (
        "https://hlc-grafana.warmteprofielengenerator.nl/api/datasources/proxy/2/query?db=tnohlc"
        "&q=SELECT%20sum(%22P_heat%22)%20FROM%20%22tot-4011%22%20WHERE%20time%20%3E%3D%201546300800000ms%20and%20time%20%3C%3D%201577836799999ms%20GROUP%20BY%20time(1h)%20fill(null)"
        "&epoch=ms" )

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:145.0) Gecko/20100101 Firefox/145.0",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "en-US,en;q=0.5",
        "Accept-Encoding": "gzip, deflate, br, zstd",
        "Referer": "https://hlc-grafana.warmteprofielengenerator.nl/d-solo/BJBoKWivz/warmtevraagprofiel-2019-4011?orgId=1&panelId=1&from=1546300800000&to=1577836799999&theme=light",
        "x-grafana-org-id": "1",
        "DNT": "1",
        "Connection": "keep-alive",
        "Sec-Fetch-Dest": "empty",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Site": "same-origin",
        "Priority": "u=4",
        "TE": "trailers"
    }

    response = requests.get(url, headers=headers)
    response.raise_for_status()
    data = response.json()

    # Query BAG buildings (pand) from PDOK WFS API as GeoJSON

    # Define your bounding box (minx, miny, maxx, maxy) in EPSG:4326 (WGS84)
    bbox = [4.354481303046308, 51.990211796688996, 4.36349681889779, 51.99831039946793]

    # WFS API returns maxFeatures=1000 by default; we need to paginate to get more
    def fetch_all_features(bbox, max_features=1000):
        features = []
        start_index = 0
        while True:
            wfs_url = (
                "https://service.pdok.nl/lv/bag/wfs/v2_0?"
                "service=WFS&version=1.1.0&request=GetFeature"
                "&typeName=bag:pand"
                "&outputFormat=application/json"
                f"&bbox={bbox[1]},{bbox[0]},{bbox[3]},{bbox[2]},urn:ogc:def:crs:EPSG::4326"
                f"&maxFeatures={max_features}"
                f"&startIndex={start_index}"
            )
            response = requests.get(wfs_url)
            response.raise_for_status()
            gdf = gpd.read_file(io.BytesIO(response.content))
            if gdf.empty:
                break
            features.append(gdf)
            if len(gdf) < max_features:
                break
            start_index += max_features
        if features:
            return gpd.GeoDataFrame(pd.concat(features, ignore_index=True), crs=features[0].crs)
        else:
            return gpd.GeoDataFrame()

    gdf = fetch_all_features(bbox)

✓ API calls to obtain geodata: 1.587s


In [5]:
# --- Demand node definition ---

with Timer("Demand node definition"):
    # Extract the heat demand series only
    series = data.get("results", [])[0].get("series", []) if "results" in data else []
    heat_df = None
    for s in series:
        name = s.get("name", "")
        columns = s.get("columns", [])
        values = s.get("values", [])
        if name == "tot-4011":
            heat_df = pd.DataFrame(values, columns=columns)

    # --- New calculation as requested ---
    if (residential_heat_demand is not None) and (heat_df is not None):
        warmtevraag_sum = residential_heat_demand["Warmtevraag"].sum()
        # Calculate share for each building
        residential_heat_demand["share"] = residential_heat_demand["Warmtevraag"] / warmtevraag_sum
        # Get max hourly heat demand from heat_df
        max_heat = heat_df["sum"].max()
        # Multiply share by max_heat
        residential_heat_demand["Peak heat demand (kW)"] = residential_heat_demand["share"] * max_heat/1000
    else:
        print("Data not available for calculation.")

    # Reproject to a projected CRS for centroid calculation, then convert centroids back to WGS84 for lon/lat
    if gdf.crs is not None and gdf.crs.to_epsg() == 4326:
        gdf_proj = gdf.to_crs(epsg=28992)  # Projected CRS for NL
        centroids_proj = gdf_proj.centroid
        centroids_wgs = gpd.GeoSeries(centroids_proj, crs=28992).to_crs(epsg=4326)
        gdf['lon'] = centroids_wgs.x
        gdf['lat'] = centroids_wgs.y
    else:
        # Fallback: use centroid for non-point geometries, but ensure output is in WGS84
        centroids = gdf.geometry.centroid
        centroids_wgs = gpd.GeoSeries(centroids, crs=gdf.crs).to_crs(epsg=4326)
        gdf['lon'] = centroids_wgs.x
        gdf['lat'] = centroids_wgs.y

    # Prepare export DataFrame with identification and coordinates
    export_cols = ['identificatie', 'lon', 'lat']
    export_df = gdf[export_cols].copy() if all(col in gdf.columns for col in export_cols) else gdf[[c for c in export_cols if c in gdf.columns]].copy()
    export_df = export_df.rename(columns={'identificatie': 'id'})

    # Merge residential_heat_demand with export_df on 'id'
    merged_df = pd.merge(residential_heat_demand, export_df, on='id', how='inner')

✓ Demand node definition: 0.062s


In [6]:
with Timer("Neighborhood heat demand plotting"):
    # Create GeoDataFrame with building polygons (preserving geometry)
    buildings_gdf = pd.merge(
        residential_heat_demand, 
        gdf[['identificatie', 'geometry']].rename(columns={'identificatie': 'id'}), 
        on='id', 
        how='inner'
    )
    buildings_gdf = gpd.GeoDataFrame(buildings_gdf, geometry='geometry', crs=gdf.crs)

    # Visualize buildings on Folium map
    if mode == 'plot':
        # Convert to WGS84 for mapping
        buildings_gdf_wgs84 = buildings_gdf.to_crs(epsg=4326)
        
        # Calculate map center from building centroids (in projected CRS to avoid warnings)
        buildings_gdf_projected = buildings_gdf.to_crs(epsg=28992)  # Project to accurate CRS
        centroids_projected = buildings_gdf_projected.geometry.centroid
        centroids_wgs84 = centroids_projected.to_crs(epsg=4326)  # Convert back to WGS84
        center_lat = centroids_wgs84.y.mean()
        center_lon = centroids_wgs84.x.mean()
        
        # Create Folium map
        buildings_map = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles="OpenStreetMap")
        
        # Add building polygons with color based on heat demand
        # Normalize heat demand for color scale
        max_demand = buildings_gdf['Peak heat demand (kW)'].max()
        min_demand = buildings_gdf['Peak heat demand (kW)'].min()
        
        for idx, row in buildings_gdf_wgs84.iterrows():
            # Calculate color intensity based on heat demand (red = high demand)
            demand = row['Peak heat demand (kW)']
            normalized = (demand - min_demand) / (max_demand - min_demand) if max_demand > min_demand else 0.5
            
            # Color from light yellow (low demand) to dark red (high demand)
            red = int(255)
            green = int(255 * (1 - normalized * 0.8))  # Reduces green as demand increases
            blue = int(255 * (1 - normalized))  # Reduces blue as demand increases
            color = f'#{red:02x}{green:02x}{blue:02x}'
            
            folium.GeoJson(
                row.geometry,
                style_function=lambda x, color=color: {
                    'fillColor': color,
                    'color': '#333333',
                    'weight': 1,
                    'fillOpacity': 0.8
                },
                popup=folium.Popup(
                    f"<b>Building ID:</b> {row['id']}<br>"
                    f"<b>Peak Heat Demand:</b> {row['Peak heat demand (kW)']:.2f} kW<br>",
                    max_width=250
                )
            ).add_to(buildings_map)
        
        # Save map
        buildings_map.save("debug/buildings_heat_demand_map.html")

✓ Neighborhood heat demand plotting: 0.648s


In [7]:
# --- Set up heat and electricity grid based on Stedin data

with Timer("Load and filter Stedin grid data"):
    # Read the shapefile into a GeoDataFrame
    stedin_heat_gdf = gpd.read_file("inputs/stedin_delft_gas_grid.geojson")
    stedin_elec_gdf = gpd.read_file("inputs/stedin_delft_elec_grid.geojson")

    # Reproject to WGS84 if needed
    if stedin_heat_gdf.crs and stedin_heat_gdf.crs.to_epsg() != 4326:
        stedin_heat_gdf = stedin_heat_gdf.to_crs(epsg=4326)

    if stedin_elec_gdf.crs and stedin_elec_gdf.crs.to_epsg() != 4326:
        stedin_elec_gdf = stedin_elec_gdf.to_crs(epsg=4326)

    # only extract coordinates for multatulibuurt

    bbox_coords = [
        (4.358862898190234, 51.989950476011565),
        (4.363513483963136, 51.99116041768696),
        (4.359959662710779, 51.997215138653104),
        (4.356868839817735, 51.996580034452485),
        (4.355168730042115, 51.995518297847504),
        (4.35819822166681, 51.9901601698577),
        (4.358862898190234, 51.989950476011565)    # repeat the first point to close the polygon
    ]

    # Create the polygon for your area of interest
    polygon = Polygon(bbox_coords)

    polygon_gdf = gpd.GeoDataFrame(index=[0], crs="EPSG:4326", geometry=[polygon])

    # Filter geometries that intersect the polygon
    stedin_heat_gdf_delft = stedin_heat_gdf[stedin_heat_gdf.geometry.intersects(polygon)]
    stedin_heat_gdf_delft = stedin_heat_gdf_delft.reset_index(drop=True)
    stedin_heat_gdf_delft['feature_name'] = [f"heat_feature{i}" for i in range(len(stedin_heat_gdf_delft))]

    stedin_elec_gdf_delft = stedin_elec_gdf[stedin_elec_gdf.geometry.intersects(polygon)].reset_index(drop=True)
    stedin_elec_gdf_delft['feature_name'] = [f"elec_feature{i}" for i in range(len(stedin_elec_gdf_delft))]

    if mode=='plot':
        # Project to a local projected CRS for accurate centroid calculation
        stedin_heat_gdf_delft_proj = stedin_heat_gdf_delft.to_crs(epsg=28992)
        centroids_proj = stedin_heat_gdf_delft_proj.geometry.centroid
        # Convert centroids back to WGS84 for mapping
        centroids_wgs = centroids_proj.to_crs(epsg=4326)
        center = [centroids_wgs.y.mean(), centroids_wgs.x.mean()]

        # Create the map
        stedin_map = folium.Map(location=center, zoom_start=13, tiles="OpenStreetMap")

        # Create FeatureGroups for each network
        heat_group = folium.FeatureGroup(name="LQ Heat/Gas Network", show=True)
        elec_group = folium.FeatureGroup(name="LV Electricity Network", show=True)

        # Add each heat feature
        for _, row in stedin_heat_gdf_delft.iterrows():
            folium.GeoJson(
                row.geometry,
                name=row['feature_name'],
                popup=folium.Popup(row['feature_name'], parse_html=True),
                style_function=lambda x: {'color': '#ff5100'}
            ).add_to(heat_group)

        # Add each electricity feature
        for _, row in stedin_elec_gdf_delft.iterrows():
            folium.GeoJson(
                row.geometry,
                name=row['feature_name'],
                popup=folium.Popup(row['feature_name'], parse_html=True),
                style_function=lambda x: {'color': '#3186cc'}
            ).add_to(elec_group)

        # Add groups to map
        heat_group.add_to(stedin_map)
        elec_group.add_to(stedin_map)

        # Add layer control for toggling
        folium.LayerControl().add_to(stedin_map)

        stedin_map.save("debug/stedin_map.html")

✓ Load and filter Stedin grid data: 3.417s


In [8]:
# --- Manually remove features from Stedin data

with Timer("Remove disconnected features"):
    # Remove feature names that are disconnected from grid
    features_to_remove_heat = ['heat_feature3','heat_feature8']
    features_to_remove_elec = ['elec_feature277','elec_feature286','elec_feature264',
                               'elec_feature261','elec_feature263','elec_feature262',
                               'elec_feature276','elec_feature252','elec_feature253',
                               'elec_feature208','elec_feature251','elec_feature250',
                               'elec_feature128','elec_feature20','elec_feature265',
                               'elec_feature270']

    # Filter out unwanted features
    stedin_heat_gdf_delft = stedin_heat_gdf_delft[~stedin_heat_gdf_delft['feature_name'].isin(features_to_remove_heat)].reset_index(drop=True).copy()
    stedin_elec_gdf_delft = stedin_elec_gdf_delft[~stedin_elec_gdf_delft['feature_name'].isin(features_to_remove_elec)].reset_index(drop=True).copy()

    if mode=='plot':
        # Project to local CRS for accurate centroid calculation (use heat network for centering)
        stedin_heat_gdf_delft_proj = stedin_heat_gdf_delft.to_crs(epsg=28992)
        centroids_proj = stedin_heat_gdf_delft_proj.geometry.centroid
        centroids_wgs = centroids_proj.to_crs(epsg=4326)
        center = [centroids_wgs.y.mean(), centroids_wgs.x.mean()]

        # Create the map
        stedin_map = folium.Map(location=center, zoom_start=13, tiles="OpenStreetMap")

        # Create FeatureGroups for each network
        heat_group = folium.FeatureGroup(name="Gas (Heat) Network", show=True)
        elec_group = folium.FeatureGroup(name="Electricity Network", show=True)

        # Add each heat feature
        for _, row in stedin_heat_gdf_delft.iterrows():
            folium.GeoJson(
                row.geometry,
                name=row['feature_name'],
                popup=folium.Popup(row['feature_name'], parse_html=True),
                style_function=lambda x: {'color': '#ff5100'}
            ).add_to(heat_group)

        # Add each electricity feature
        for _, row in stedin_elec_gdf_delft.iterrows():
            folium.GeoJson(
                row.geometry,
                name=row['feature_name'],
                popup=folium.Popup(row['feature_name'], parse_html=True),
                style_function=lambda x: {'color': '#3186cc'}
            ).add_to(elec_group)

        # Add groups to map
        heat_group.add_to(stedin_map)
        elec_group.add_to(stedin_map)

        # Add layer control for toggling
        folium.LayerControl().add_to(stedin_map)

        stedin_map.save("debug/stedin_map.html")

✓ Remove disconnected features: 0.762s


In [9]:
# --- Clean up electricity grid

with Timer("Clean up electricity grid topology"):
    # --- Parameters ---
    simplify_tolerance = 0.000001  # Adjust for your use case (in degrees)
    snap_tolerance = 0.000001     # Snap threshold (in degrees)

    # 1. Simplify all geometries
    stedin_elec_gdf_delft['geometry'] = stedin_elec_gdf_delft['geometry'].apply(
        lambda geom: geom.simplify(simplify_tolerance, preserve_topology=True)
    )

    # 2. Snap all features together

    all_geoms = list(stedin_elec_gdf_delft.geometry)
    # Snap each geometry to all others (pairwise, no unary_union)
    snapped_geoms = []
    for i, geom in enumerate(all_geoms):
        snapped = geom
        for j, other in enumerate(all_geoms):
            if i != j:
                snapped = snap(snapped, other, snap_tolerance)
        snapped_geoms.append(snapped)

    stedin_elec_gdf_delft['geometry'] = snapped_geoms

    if mode=='plot':
        # Project to local CRS for accurate centroid calculation (use heat network for centering)
        stedin_heat_gdf_delft_proj = stedin_heat_gdf_delft.to_crs(epsg=28992)
        centroids_proj = stedin_heat_gdf_delft_proj.geometry.centroid
        centroids_wgs = centroids_proj.to_crs(epsg=4326)
        center = [centroids_wgs.y.mean(), centroids_wgs.x.mean()]

        # Create the map
        stedin_map = folium.Map(location=center, zoom_start=13, tiles="OpenStreetMap")

        # Create FeatureGroups for each network
        heat_group = folium.FeatureGroup(name="Gas (Heat) Network", show=True)
        elec_group = folium.FeatureGroup(name="Electricity Network", show=True)

        # Add each heat feature
        for _, row in stedin_heat_gdf_delft.iterrows():
            folium.GeoJson(
                row.geometry,
                name=row['feature_name'],
                popup=folium.Popup(row['feature_name'], parse_html=True),
                style_function=lambda x: {'color': '#ff5100'}
            ).add_to(heat_group)

        # Add each electricity feature
        for _, row in stedin_elec_gdf_delft.iterrows():
            folium.GeoJson(
                row.geometry,
                name=row['feature_name'],
                popup=folium.Popup(row['feature_name'], parse_html=True),
                style_function=lambda x: {'color': '#3186cc'}
            ).add_to(elec_group)

        # Add groups to map
        heat_group.add_to(stedin_map)
        elec_group.add_to(stedin_map)

        # Add layer control for toggling
        folium.LayerControl().add_to(stedin_map)

        stedin_map.save("debug/stedin_map.html")

✓ Clean up electricity grid topology: 1.736s


In [10]:
# --- Create transmission nodes and interpolate for heat and electricity networks separately ---

with Timer("Create and interpolate transmission nodes"):
    #stedin_elec_gdf_delft=stedin_heat_gdf_delft

    spacing_m=2.5

    # --- Helper: Haversine distance in km ---
    def haversine_distance(lat1, lon1, lat2, lon2):
        R = 6371  # Earth radius in km
        phi1, phi2 = np.radians(lat1), np.radians(lat2)
        dphi = np.radians(lat2 - lat1)
        dlambda = np.radians(lon2 - lon1)
        a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
        return 2 * R * np.arcsin(np.sqrt(a))

    # --- Interpolation function ---
    def interpolate_line(lat1, lon1, lat2, lon2, spacing_m):
        total_dist_km = haversine_distance(lat1, lon1, lat2, lon2)
        total_dist_m = total_dist_km * 1000
        if total_dist_m < 1e-6:
            return [(lat1, lon1), (lat2, lon2)]
        n_points = int(np.floor(total_dist_m / spacing_m))
        if n_points < 1:
            return [(lat1, lon1), (lat2, lon2)]
        lats = np.linspace(lat1, lat2, n_points + 2)
        lons = np.linspace(lon1, lon2, n_points + 2)
        return list(zip(lats, lons))

    def extract_and_interpolate_nodes(gdf, spacing_m):
        # Collect all unique corner points from all geometries
        corner_points = set()
        for geom in gdf.geometry:
            if geom.geom_type == "LineString":
                for coord in geom.coords:
                    corner_points.add(coord)
            elif geom.geom_type == "MultiLineString":
                for part in geom.geoms:
                    for coord in part.coords:
                        corner_points.add(coord)

        # Create a GeoDataFrame of nodes
        nodes_gdf = gpd.GeoDataFrame(
            geometry=[Point(x, y) for x, y in corner_points],
            crs=gdf.crs
        )
        nodes_gdf["lon"] = nodes_gdf.geometry.x
        nodes_gdf["lat"] = nodes_gdf.geometry.y

        # Interpolate along all segments
        interpolated_points = []
        for geom in gdf.geometry:
            if geom.geom_type == "LineString":
                coords = list(geom.coords)
                for i in range(len(coords) - 1):
                    lat1, lon1 = coords[i][1], coords[i][0]
                    lat2, lon2 = coords[i+1][1], coords[i+1][0]
                    points = interpolate_line(lat1, lon1, lat2, lon2, spacing_m=spacing_m)
                    interpolated_points.extend(points)
            elif geom.geom_type == "MultiLineString":
                for part in geom.geoms:
                    coords = list(part.coords)
                    for i in range(len(coords) - 1):
                        lat1, lon1 = coords[i][1], coords[i][0]
                        lat2, lon2 = coords[i+1][1], coords[i+1][0]
                        points = interpolate_line(lat1, lon1, lat2, lon2, spacing_m=spacing_m)
                        interpolated_points.extend(points)

        # Remove duplicates and create GeoDataFrame
        unique_points = list({(round(lat, 7), round(lon, 7)) for lat, lon in interpolated_points})
        interp_gdf = gpd.GeoDataFrame(
            geometry=[Point(lon, lat) for lat, lon in unique_points],
            crs="EPSG:4326"
        )
        interp_gdf["lat"] = interp_gdf.geometry.y
        interp_gdf["lon"] = interp_gdf.geometry.x

        return nodes_gdf, interp_gdf

    # Extract and interpolate heat (gas) nodes
    heat_nodes_gdf, heat_interp_gdf = extract_and_interpolate_nodes(stedin_heat_gdf_delft, spacing_m)

    # Extract and interpolate electricity nodes
    elec_nodes_gdf, elec_interp_gdf = extract_and_interpolate_nodes(stedin_elec_gdf_delft, spacing_m)

✓ Create and interpolate transmission nodes: 0.476s


In [11]:
# --- Import csv files from inputs folder

with Timer("Import CSV files from inputs folder"):
    inputs_folder = "inputs"

    # Dictionary to hold DataFrames
    input_dataframes = {}

    # Read all CSV files in the folder
    for filename in os.listdir(inputs_folder):
        if filename.endswith(".csv"):
            # Create a descriptive variable name from the filename
            var_name = filename.replace(".csv", "").replace("-", "_").replace(" ", "_")
            df = pd.read_csv(os.path.join(inputs_folder, filename))
            input_dataframes[var_name] = df
            globals()[var_name] = df  # Optionally set as a variable in the global namespace

✓ Import CSV files from inputs folder: 0.010s


In [12]:
# --- Add demand and transmission nodes

with Timer("Add demand and transmission nodes"):
    demand_nodes=merged_df
    demand_nodes['id'] = 'D' + demand_nodes['id'].astype(str)

    heat_trans_nodes = heat_interp_gdf.copy()
    heat_trans_nodes["id"] = [f"LQHtransmission{i+1}" for i in range(len(heat_trans_nodes))]

    elec_trans_nodes = elec_interp_gdf.copy()
    elec_trans_nodes["id"] = [f"LVEtransmission{i+1}" for i in range(len(elec_trans_nodes))]

    demand_techs = pd.DataFrame({
        "nodes": demand_nodes["id"],
        "techs": "demand_LQ_heat",
        "parameters": "sink_use_equals",
        "timesteps": "",
        "2050/01/01 00:00": demand_nodes["Peak heat demand (kW)"]
    })

    # 2. Prepare transmission node tech assignments (value = 0)
    heat_trans_techs = pd.DataFrame({
        "nodes": heat_trans_nodes["id"],
        "techs": "demand_LQ_heat",
        "parameters": "sink_use_equals",
        "timesteps": "",
        "2050/01/01 00:00": 0
    })

    elec_trans_techs = pd.DataFrame({
        "nodes": elec_trans_nodes["id"],
        "techs": "demand_electricity",
        "parameters": "sink_use_equals",
        "timesteps": "",
        "2050/01/01 00:00": 0
    })

    # 3. Concatenate both and append to warmtenet_nodes_techs
    transmission_techs = pd.concat([heat_trans_techs, elec_trans_techs], ignore_index=True)
    new_techs = pd.concat([demand_techs, transmission_techs], ignore_index=True)
    old_techs = pd.concat([warmtenet_nodes_techs, MV_LV_transformer_nodes_techs], ignore_index=True)
    nodes_techs = pd.concat([old_techs, new_techs], ignore_index=True)

    # Prepare DataFrames for demand and transmission node coordinates
    demand_coords = demand_nodes[["id", "lon", "lat"]].copy().rename(columns={"id": "nodes", "lon": "longitude", "lat": "latitude"})
    heat_trans_coords = heat_trans_nodes[["id", "lon", "lat"]].copy().rename(columns={"id": "nodes", "lon": "longitude", "lat": "latitude"})
    elec_trans_coords = elec_trans_nodes[["id", "lon", "lat"]].copy().rename(columns={"id": "nodes", "lon": "longitude", "lat": "latitude"})

    # Concatenate both
    trans_coords = pd.concat([heat_trans_coords, elec_trans_coords], ignore_index=True)
    new_coords = pd.concat([demand_coords, trans_coords], ignore_index=True)
    old_coords = pd.concat([warmtenet_nodes_coordinates, MV_LV_transformer_nodes_coordinates], ignore_index=True)
    nodes_coordinates= pd.concat([old_coords, new_coords], ignore_index=True)

✓ Add demand and transmission nodes: 0.012s


In [13]:
# --- Add links between transmission nodes ---

with Timer("Add links between transmission nodes"):
    # 2. Build a mapping from (lon, lat) to node ID
    coord_to_id_heat = {(round(row.lon, 7), round(row.lat, 7)): row.id for _, row in heat_trans_nodes.iterrows()}

    # 3. Collect links between connected Stedin nodes
    heat_links = []
    for geom in stedin_heat_gdf_delft.geometry:
        if geom.geom_type == "LineString":
            coords = list(geom.coords)
            for i in range(len(coords) - 1):
                lat1, lon1 = coords[i][1], coords[i][0]
                lat2, lon2 = coords[i+1][1], coords[i+1][0]
                points = interpolate_line(lat1, lon1, lat2, lon2, spacing_m=spacing_m)
                for j in range(len(points) - 1):
                    pt1 = (round(points[j][1], 7), round(points[j][0], 7))
                    pt2 = (round(points[j+1][1], 7), round(points[j+1][0], 7))
                    if pt1 in coord_to_id_heat and pt2 in coord_to_id_heat:
                        node_from = coord_to_id_heat[pt1]
                        node_to = coord_to_id_heat[pt2]
                        # Example: create a heat link (customize as needed)
                        link_name_heat = f"{node_from}_to_{node_to}_heat"
                        heat_links.append({
                            "techs": link_name_heat,
                            "color": "#823740",
                            "name": "LQ heat distribution main",
                            "base_tech": "transmission",
                            "flow_cap_max": 10000,
                            "flow_out_eff_per_distance": 0.99,
                            "lifetime": 20,
                            "link_to": node_to,
                            "link_from": node_from
                        })

    # 2. Build a mapping from (lon, lat) to node ID
    coord_to_id_elec = {(round(row.lon, 7), round(row.lat, 7)): row.id for _, row in elec_trans_nodes.iterrows()}

    # 3. Collect links between connected Stedin nodes
    elec_links = []
    for geom in stedin_elec_gdf_delft.geometry:
        if geom.geom_type == "LineString":
            coords = list(geom.coords)
            for i in range(len(coords) - 1):
                lat1, lon1 = coords[i][1], coords[i][0]
                lat2, lon2 = coords[i+1][1], coords[i+1][0]
                points = interpolate_line(lat1, lon1, lat2, lon2, spacing_m=spacing_m)
                for j in range(len(points) - 1):
                    pt1 = (round(points[j][1], 7), round(points[j][0], 7))
                    pt2 = (round(points[j+1][1], 7), round(points[j+1][0], 7))
                    if pt1 in coord_to_id_elec and pt2 in coord_to_id_elec:
                        node_from = coord_to_id_elec[pt1]
                        node_to = coord_to_id_elec[pt2]
                        link_name_elec = f"{node_from}_to_{node_to}_electricity"
                        elec_links.append({
                            "techs": link_name_elec,
                            "color": "#3186cc",
                            "name": "LV electricity distribution main",
                            "base_tech": "transmission",
                            "flow_cap_max": 10000,
                            "flow_out_eff_per_distance": 0.99,
                            "lifetime": 20,
                            "link_to": node_to,
                            "link_from": node_from
                        })

    new_links=pd.concat([pd.DataFrame(heat_links), pd.DataFrame(elec_links)], ignore_index=True)
    links_techs= pd.concat([warmtenet_links_techs, new_links], ignore_index=True)

✓ Add links between transmission nodes: 0.798s


In [14]:
# --- Connect demand nodes, substations, and transformers to distribution network (OPTIMIZED) ---

with Timer("Connect nodes to distribution network"):
    # 1. Get coordinates of substation_multatulibuurt
    substation_row = warmtenet_nodes_coordinates[warmtenet_nodes_coordinates['nodes'] == 'substation_multatulibuurt']
    if substation_row.empty:
        raise ValueError("substation_multatulibuurt not found in warmtenet_nodes_coordinates")
    sub_lon = substation_row.iloc[0]['longitude']
    sub_lat = substation_row.iloc[0]['latitude']

    # 2. Get transmission nodes and their coordinates (heat and electricity)
    heat_trans_nodes_coords = heat_trans_nodes[['id', 'lon', 'lat']].copy()
    elec_trans_nodes_coords = elec_trans_nodes[['id', 'lon', 'lat']].copy()

    # 3. Substation to nearest heat node (vectorized)
    heat_lats = heat_trans_nodes_coords['lat'].values
    heat_lons = heat_trans_nodes_coords['lon'].values
    dists = haversine_distance(sub_lat, sub_lon, heat_lats, heat_lons)
    nearest_heat_idx = np.argmin(dists)
    nearest_heat_id = heat_trans_nodes_coords.iloc[nearest_heat_idx]['id']

    new_link = {
        "techs": f"substation_multatulibuurt_to_{nearest_heat_id}_heat",
        "color": "#823740",
        "name": "LQ heat distribution main",
        "base_tech": "transmission",
        "flow_cap_max": 10000,
        "flow_out_eff_per_distance": 0.99,
        "lifetime": 20,
        "link_to": nearest_heat_id,
        "link_from": "substation_multatulibuurt"
    }
    links_techs = pd.concat([links_techs, pd.DataFrame([new_link])], ignore_index=True)

    # 4. Transformers to nearest electricity node (vectorized)
    mv_lats = MV_LV_transformer_nodes_coordinates['latitude'].values
    mv_lons = MV_LV_transformer_nodes_coordinates['longitude'].values
    elec_lats = elec_trans_nodes_coords['lat'].values
    elec_lons = elec_trans_nodes_coords['lon'].values

    # Broadcasting: (n_transformers, 1) - (1, n_elec_nodes) = (n_transformers, n_elec_nodes)
    dists = haversine_distance(mv_lats[:, None], mv_lons[:, None], elec_lats[None, :], elec_lons[None, :])
    nearest_idxs = np.argmin(dists, axis=1)

    # Build all transformer links at once
    mv_nodes = MV_LV_transformer_nodes_coordinates['nodes'].values
    nearest_elec_ids = elec_trans_nodes_coords.iloc[nearest_idxs]['id'].values

    new_elec_links = pd.DataFrame({
        "techs": [f"{mv}_to_{elec}_electricity" for mv, elec in zip(mv_nodes, nearest_elec_ids)],
        "color": "#3186cc",
        "name": "LV electricity distribution main",
        "base_tech": "transmission",
        "flow_cap_max": 10000,
        "flow_out_eff_per_distance": 0.99,
        "lifetime": 20,
        "link_to": nearest_elec_ids,
        "link_from": mv_nodes
    })
    links_techs = pd.concat([links_techs, new_elec_links], ignore_index=True)

    # 5. Demand nodes to nearest heat and electricity nodes (vectorized)
    demand_lats = demand_nodes['lat'].values
    demand_lons = demand_nodes['lon'].values

    # Heat distances (broadcasting)
    dists_heat = haversine_distance(demand_lats[:, None], demand_lons[:, None], heat_lats[None, :], heat_lons[None, :])
    nearest_heat_idxs = np.argmin(dists_heat, axis=1)

    # Elec distances (broadcasting)
    dists_elec = haversine_distance(demand_lats[:, None], demand_lons[:, None], elec_lats[None, :], elec_lons[None, :])
    nearest_elec_idxs = np.argmin(dists_elec, axis=1)

    # Get IDs
    demand_ids = demand_nodes['id'].values
    nearest_heat_ids = heat_trans_nodes_coords.iloc[nearest_heat_idxs]['id'].values
    nearest_elec_ids = elec_trans_nodes_coords.iloc[nearest_elec_idxs]['id'].values

    # Build heat links DataFrame
    heat_links_df = pd.DataFrame({
        "techs": [f"{d}_to_{h}_heat" for d, h in zip(demand_ids, nearest_heat_ids)],
        "color": "#823740",
        "name": "LQ heat distribution secondary",
        "base_tech": "transmission",
        "flow_cap_max": 10000,
        "flow_out_eff_per_distance": 0.99,
        "lifetime": 20,
        "link_to": nearest_heat_ids,
        "link_from": demand_ids
    })

    # Build elec links DataFrame
    elec_links_df = pd.DataFrame({
        "techs": [f"{d}_to_{e}_electricity" for d, e in zip(demand_ids, nearest_elec_ids)],
        "color": "#3186cc",
        "name": "LV electricity distribution secondary",
        "base_tech": "transmission",
        "flow_cap_max": 10000,
        "flow_out_eff_per_distance": 0.99,
        "lifetime": 20,
        "link_to": nearest_elec_ids,
        "link_from": demand_ids
    })

    # Concatenate all at once
    links_techs = pd.concat([links_techs, heat_links_df, elec_links_df], ignore_index=True)
    links_techs = links_techs.drop_duplicates(subset=['link_from', 'link_to', 'name'])

✓ Connect nodes to distribution network: 0.203s


In [15]:
with Timer("Create link carrier and cost DataFrames"):
    # Create an empty copy of warmtenet_links_carriers with the same columns and dtypes
    links_LQ_heat = warmtenet_links_carriers.iloc[0:0].copy()

    # Select all 'techs' values from links_techs where 'name' is "LQ heat distribution"
    lq_heat_techs = links_techs.loc[links_techs['name'].str.contains("LQ heat distribution", na=False), 'techs']

    # Create a DataFrame with these techs and fill all other columns with 1
    new_rows = pd.DataFrame(1, index=range(len(lq_heat_techs)), columns=links_LQ_heat.columns)
    new_rows['techs'] = lq_heat_techs.values

    # Concatenate to links_LQ_heat
    links_LQ_heat = pd.concat([links_LQ_heat, new_rows], ignore_index=True)

    # Select all 'techs' values from links_techs where 'name' is "LV electricity distribution"
    lv_elec_techs = links_techs.loc[links_techs['name'].str.contains("LV electricity distribution", na=False), 'techs']

    # Create a DataFrame with these techs and fill all other columns with 1
    new_rows_elec = pd.DataFrame(1, index=range(len(lv_elec_techs)), columns=links_LQ_heat.columns)
    new_rows_elec['techs'] = lv_elec_techs.values

    # Concatenate to a new DataFrame, e.g., links_LV_electricity
    links_electricity = links_LQ_heat.iloc[0:0].copy()
    links_electricity = pd.concat([links_electricity, new_rows_elec], ignore_index=True)

    # Create a DataFrame with 'techs' from links_techs and a constant value in 'cost_flow_cap_per_distance'
    links_costs = pd.DataFrame({
        'techs': links_techs['techs'],
        'cost_flow_cap_per_distance': 100
    })

✓ Create link carrier and cost DataFrames: 0.013s


In [16]:
# --- Save dataframes to csv files

with Timer("Save DataFrames to CSV files"):
    warmtenet_links_carriers.to_csv('data_tables/warmtenet_links_carriers.csv', index=False)
    nodes_techs.to_csv('data_tables/nodes_techs.csv', index=False)
    nodes_coordinates.to_csv('data_tables/nodes_coordinates.csv', index=False)
    links_techs.to_csv('data_tables/links_techs.csv', index=False)
    links_LQ_heat.to_csv('data_tables/links_LQ_heat.csv', index=False)
    links_electricity.to_csv('data_tables/links_electricity.csv', index=False)
    links_costs.to_csv('data_tables/links_costs.csv', index=False)

✓ Save DataFrames to CSV files: 0.220s


In [17]:
# --- Visualize network ---

with Timer("Visualize network"):
    if mode=='plot':
        # 1. Prepare node coordinates DataFrame
        node_coords = nodes_coordinates.set_index('nodes')

        # 2. Prepare links DataFrame
        if 'link_from' not in links_techs.columns or 'link_to' not in links_techs.columns:
            links_techs = links_techs.copy()
            links_techs[['link_from', 'link_to']] = links_techs['techs'].str.extract(r'^(.*?)_to_(.*?)_')

        # 3. Create a Folium map centered on the mean coordinates
        center_lat = node_coords['latitude'].mean()
        center_lon = node_coords['longitude'].mean()
        network_map = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles='OpenStreetMap')

        # 4. Create FeatureGroups for heat and electricity links
        heat_links_group = folium.FeatureGroup(name="Heat Links", show=True)
        elec_links_group = folium.FeatureGroup(name="Electricity Links", show=True)

        # 5. Add nodes as circle markers
        for node, row in node_coords.iterrows():
            folium.CircleMarker(
                location=[row['latitude'], row['longitude']],
                radius=4,
                color='red',
                fill=True,
                fill_color='red',
                fill_opacity=0.7,
                popup=f"Node: {node}"
            ).add_to(network_map)

        # 6. Add links as lines, separated by type
        for _, link in links_techs.iterrows():
            from_node = link['link_from']
            to_node = link['link_to']
            if from_node in node_coords.index and to_node in node_coords.index:
                from_lat, from_lon = node_coords.loc[from_node, ['latitude', 'longitude']]
                to_lat, to_lon = node_coords.loc[to_node, ['latitude', 'longitude']]
                if "LQ heat distribution" in link['name']:
                    folium.PolyLine(
                        locations=[[from_lat, from_lon], [to_lat, to_lon]],
                        color='#ff5100',
                        weight=2,
                        opacity=0.7,
                        popup=f"{from_node} → {to_node}"
                    ).add_to(heat_links_group)
                elif "LV electricity distribution" in link['name']:
                    folium.PolyLine(
                        locations=[[from_lat, from_lon], [to_lat, to_lon]],
                        color='#3186cc',
                        weight=2,
                        opacity=0.7,
                        popup=f"{from_node} → {to_node}"
                    ).add_to(elec_links_group)

        # 7. Add groups to map and layer control
        heat_links_group.add_to(network_map)
        elec_links_group.add_to(network_map)
        folium.LayerControl().add_to(network_map)

        # 8. Save and display the map
        network_map.save('debug/network_map.html')

✓ Visualize network: 24.482s


In [18]:
# --- Scenario creation and model running ---

with Timer("Create scenario and run model"):
    #calliope.set_log_verbosity("DEBUG", include_solver_output=True)

    # --- Scenario definition ---
    scenario = 'full_electrification'  # Options: 'full_electrification' or 'district_heating'

    if scenario == 'full_electrification':
        
        # 1. Load the nodes data that was just created in the previous cell
        nodes_coords = pd.read_csv('data_tables/nodes_coordinates.csv')
        demand_nodes = nodes_coords[nodes_coords['nodes'].str.startswith('D')]['nodes']

        # 2. Read the model.yaml file
        # Using ruamel.yaml to preserve comments and structure
        yaml = YAML()
        yaml_path = 'district_heating_model.yaml'
        with open(yaml_path, 'r') as f:
            model_config = yaml.load(f)
        
        # 3. Add the heat pump technology to each demand node in the YAML structure
        # Create the top-level 'nodes' key if it doesn't exist
        if 'nodes' not in model_config:
            model_config['nodes'] = {}
        
        # For each demand node, add an entry to allow 'heat_pump' to be built
        for node_name in demand_nodes:
            if node_name not in model_config['nodes']:
                model_config['nodes'][node_name] = {}
            if 'techs' not in model_config['nodes'][node_name]:
                model_config['nodes'][node_name]['techs'] = {}
            # Adding the technology with an empty dictionary is enough to make it available
            model_config['nodes'][node_name]['techs']['heat_pump'] = {}
            
        # 4. Write the updated configuration back to the model.yaml file
        new_yaml_path = 'electrification_model.yaml'
        with open(new_yaml_path, 'w') as f:
            yaml.dump(model_config, f)

        # 5. Deactivate district heating links
        links_techs = links_techs[~links_techs['name'].str.contains("LQ heat distribution", na=False)].reset_index(drop=True)
        links_techs.to_csv('data_tables/links_techs.csv', index=False)

        links_LQ_heat = links_LQ_heat[~links_LQ_heat['techs'].str.endswith('_heat')].reset_index(drop=True)
        links_LQ_heat.to_csv('data_tables/links_LQ_heat.csv', index=False)

        links_costs = links_costs[~links_costs['techs'].str.endswith('_heat')].reset_index(drop=True)
        links_costs.to_csv('data_tables/links_costs.csv', index=False)

        # Deactivate electricity distribution nodes
        nodes_techs = nodes_techs[~nodes_techs['nodes'].str.startswith('LQHtransmission')].reset_index(drop=True)
        nodes_techs.to_csv('data_tables/nodes_techs.csv', index=False)
        
        nodes_coordinates = nodes_coordinates[~nodes_coordinates['nodes'].str.startswith('LQHtransmission')].reset_index(drop=True)
        nodes_coordinates.to_csv('data_tables/nodes_coordinates.csv', index=False)
        
        model = calliope.read_yaml("electrification_model.yaml")

    elif scenario == 'district_heating':

        # Deactivate electricity distribution links
        links_techs = links_techs[~links_techs['name'].str.contains("LV electricity distribution", na=False)].reset_index(drop=True)
        links_techs.to_csv('data_tables/links_techs.csv', index=False)

        links_electricity = links_electricity[~links_electricity['techs'].str.endswith('_electricity')].reset_index(drop=True)
        links_electricity.to_csv('data_tables/links_electricity.csv', index=False)

        links_costs = links_costs[~links_costs['techs'].str.endswith('_electricity')].reset_index(drop=True)
        links_costs.to_csv('data_tables/links_costs.csv', index=False)

        # Deactivate electricity distribution nodes
        nodes_techs = nodes_techs[~nodes_techs['nodes'].str.startswith('LVEtransmission')].reset_index(drop=True)
        nodes_techs.to_csv('data_tables/nodes_techs.csv', index=False)
        
        nodes_coordinates = nodes_coordinates[~nodes_coordinates['nodes'].str.startswith('LVEtransmission')].reset_index(drop=True)
        nodes_coordinates.to_csv('data_tables/nodes_coordinates.csv', index=False)


        model = calliope.read_yaml("district_heating_model.yaml")

✓ Create scenario and run model: 319.907s


In [19]:
#model.inputs

In [20]:
# --- Building and solving of Calliope model ---
with Timer("Build Calliope model"):
    model.build()

with Timer("Solve Calliope model"):
    model.solve()

✓ Build Calliope model: 820.446s
✓ Solve Calliope model: 121.265s


In [21]:
# --- Calliope model results visualization ---

with Timer("Visualize Calliope results"):
    if mode=='plot':
        df_coords = model.inputs[["latitude", "longitude"]].to_dataframe().reset_index()

        df_capacity = (
            model.results.flow_cap.where(model.inputs.base_tech == "transmission")
            .to_series()
            .where(lambda x: x != 0)
            .dropna()
            .to_frame("Flow capacity (kW)")
            .reset_index()
        )

        # Define distribution and transmission dataframes for plotting
        df_capacity_coords = pd.merge(df_coords, df_capacity, left_on="nodes", right_on="nodes").sort_values(by=['techs'])

        # Extract link information from techs column (format: "node_from_to_node_to")
        df_links = df_capacity_coords.copy()

        # Split the techs column to get link_from and link_to
        df_links[['link_from', 'link_to_carrier']] = df_links['techs'].str.rsplit('_to_', n=1, expand=True)
        df_links['link_to'] = df_links['link_to_carrier'].str.replace(r'_(heat|electricity|HQ_heat|LQ_heat)$', '', regex=True)
        df_links['link_from'] = df_links['link_from'].str.replace(r'_(heat|electricity|HQ_heat|LQ_heat)$', '', regex=True)

        # Merge with df_coords twice to get both from and to coordinates
        # First merge for "from" coordinates
        df_links = df_links.merge(
            df_coords[['nodes', 'latitude', 'longitude']],
            left_on='link_from',
            right_on='nodes',
            how='left',
            suffixes=('', '_from')
        )
        df_links = df_links.rename(columns={'latitude': 'lat_from', 'longitude': 'lon_from'})

        # Second merge for "to" coordinates
        df_links = df_links.merge(
            df_coords[['nodes', 'latitude', 'longitude']],
            left_on='link_to',
            right_on='nodes',
            how='left',
            suffixes=('_temp', '_to')
        )
        df_links = df_links.rename(columns={'latitude': 'lat_to', 'longitude': 'lon_to'})

        # Clean up duplicate columns
        df_links = df_links.drop(columns=['nodes_temp', 'nodes_to'], errors='ignore')

        # Create a Folium map centered on your data
        center_lat = df_coords['latitude'].mean()
        center_lon = df_coords['longitude'].mean()

        map_fig = folium.Map(
            location=[center_lat, center_lon],
            zoom_start=16,
            tiles='OpenStreetMap'
        )


        # Create FeatureGroups for different layers
        demand_group = folium.FeatureGroup(name="Demand Nodes", show=True).add_to(map_fig)
        supply_heat_group = folium.FeatureGroup(name="Supply Heat Nodes", show=True).add_to(map_fig)
        supply_elec_group = folium.FeatureGroup(name="Supply Electricity Nodes", show=True).add_to(map_fig)
        transmission_heat_group = folium.FeatureGroup(name="Heat Transmission Nodes", show=True).add_to(map_fig)
        distribution_heat_group = folium.FeatureGroup(name="Heat Transmission Nodes", show=True).add_to(map_fig)
        HQ_heat_link_group = folium.FeatureGroup(name="HQ Heat Links", show=True).add_to(map_fig)
        LQ_heat_link_group = folium.FeatureGroup(name="LQ Heat Links", show=True).add_to(map_fig)
        distribution_electricity_group = folium.FeatureGroup(name="Electricity Transmission Nodes", show=True).add_to(map_fig)
        electricity_link_group = folium.FeatureGroup(name="Electricity Links", show=True).add_to(map_fig)
        substation_group = folium.FeatureGroup(name="Substations", show=True).add_to(map_fig)
        building_group = folium.FeatureGroup(name="Buildings", show=True).add_to(map_fig)

        # Add lines for each link to the 'link_group' FeatureGroup
        for idx, row in df_links.iterrows():
            if row['carriers'] == 'HQ_heat':
                color = 'red'
                target_group = HQ_heat_link_group
                weight = 4
            elif row['carriers'] == 'LQ_heat':
                color = "#ff5100"
                target_group = LQ_heat_link_group
                weight = 2
            else:
                color = 'blue'
                target_group = electricity_link_group
                weight = 2
                
            folium.PolyLine(
                locations=[[row['lat_from'], row['lon_from']], [row['lat_to'], row['lon_to']]],
                color=color,
                weight=weight,
                opacity=1,
                popup=f"<b>{row['techs']}</b><br>From: {row['link_from']}<br>To: {row['link_to']}<br>Capacity: {row['Flow capacity (kW)']} kW"
            ).add_to(target_group)


        # Add node markers to their respective FeatureGroups
        for idx, row in df_capacity_coords.iterrows():
            node_name = row['nodes']
            
            # Determine node type and styling
            if node_name.startswith('geothermie'):
                color = '#2ecc71' 
                radius = 5
                node_type = 'Supply heat'
                target_group = supply_heat_group
            elif node_name.startswith('MV'):
                color = "#2e38cc" 
                radius = 1
                node_type = 'Supply electricity'
                target_group = supply_elec_group
            elif node_name.startswith('D'):
                color = "#ff9100"  
                radius = 3
                node_type = 'Demand'
                target_group = demand_group
            elif node_name.startswith('warmtenet'):
                color = "#94d3ae" 
                radius = 1
                node_type = 'Transmission heat'
                target_group = transmission_heat_group
            elif node_name.startswith('LQHtransmission'):
                color = "#94d3ae" 
                radius = 1
                node_type = 'Distribution heat'
                target_group = distribution_heat_group
            elif node_name.startswith('substation'):
                color = "#ff3300"  
                radius = 3
                node_type = 'Heat substation'
                target_group = substation_group
            else:  
                color = "#7076cc"  
                radius = 1
                node_type = 'Distribution electricity'
                target_group = distribution_electricity_group
            
            folium.CircleMarker(
                location=[row['latitude'], row['longitude']],
                radius=radius,
                popup=f"<b>{row['nodes']}</b> ({node_type})<br>Capacity: {row['Flow capacity (kW)']} kW",
                color=color,
                fill=True,
                fillColor=color,
                fillOpacity=1,
                weight=2
            ).add_to(target_group) # Add to the correct group

        # Add building polygons with color based on heat demand
        # Normalize heat demand for color scale
        max_demand = buildings_gdf['Peak heat demand (kW)'].max()
        min_demand = buildings_gdf['Peak heat demand (kW)'].min()
        
        for idx, row in buildings_gdf_wgs84.iterrows():
            # Calculate color intensity based on heat demand (red = high demand)
            demand = row['Peak heat demand (kW)']
            normalized = (demand - min_demand) / (max_demand - min_demand) if max_demand > min_demand else 0.5
            
            # Color from light yellow (low demand) to dark red (high demand)
            red = int(255)
            green = int(255 * (1 - normalized*0.9))  # Reduces green as demand increases
            blue = int(255 * (1 - normalized*0.9))  # Reduces blue as demand increases
            color = f'#{red:02x}{green:02x}{blue:02x}'
            
            folium.GeoJson(
                row.geometry,
                style_function=lambda x, color=color: {
                    'fillColor': color,
                    'color': '#333333',
                    'weight': 1,
                    'fillOpacity': 0.8
                },
                popup=folium.Popup(
                    f"<b>Building ID:</b> {row['id']}<br>"
                    f"<b>Peak Heat Demand:</b> {row['Peak heat demand (kW)']:.2f} kW<br>",
                    max_width=250
                )
            ).add_to(building_group)

        # --- Add the LayerControl to the map ---
        # This creates the toggle switch in the top-right corner
        folium.LayerControl().add_to(map_fig)

        # Display the map
        map_fig.save("outputs/system_map.html")

✓ Visualize Calliope results: 14.114s


In [22]:
# --- Bill of materials export ---

with Timer("Export bill of materials"):
    # For each item to be exported, find its name in inputs, merge with capacity data, and export to dataframe
    tech_names = model.inputs.name.to_series().dropna()
    tech_distances = model.inputs.distance.to_series().dropna()

    total_flow_out = (
        model.results.flow_out
        .sum(dim=["nodes", "carriers", "timesteps"], min_count=1)
        .to_series()
        .dropna()
    )

    export_df = pd.DataFrame({
        'name': tech_names,
        'capacity_kw': total_flow_out,
        'distance_m': tech_distances*1000
    })

    final_export_df = export_df[export_df['capacity_kw'] > 0].sort_values(by=['name', 'capacity_kw'], ascending=[True, False]).reset_index(drop=True)

    # For each item in the export dataframe, multiply capacity by some environmental impact factor, and add environmental impact column


    final_export_df.to_csv('outputs/bill_of_materials.csv', index=False)

final_export_df.head()

✓ Export bill of materials: 1.624s


,name,capacity_kw,distance_m
0,Air-to-air heat pump,4588.115872,NaN
1,LV electricity distribution main,999.990625,0.932778
2,LV electricity distribution main,999.985927,1.400260
3,LV electricity distribution main,999.985096,1.482975
4,LV electricity distribution main,999.983710,0.137883


In [23]:
# --- Timing Summary ---

# Display comprehensive timing analysis
if execution_times:
    timing_df = pd.DataFrame({
        'Operation': list(execution_times.keys()),
        'Time (s)': list(execution_times.values())
    })
    timing_df['% of Total'] = (timing_df['Time (s)'] / timing_df['Time (s)'].sum() * 100).round(2)
    timing_df = timing_df.sort_values('Time (s)', ascending=False)
    
    print("\n" + "="*80)
    print("EXECUTION TIME SUMMARY")
    print("="*80)
    print(timing_df.to_string(index=False))
    print("="*80)
    print(f"Total Execution Time: {timing_df['Time (s)'].sum():.3f}s")
    print("="*80)
else:
    print("No timing data collected. Run the notebook cells to collect timing information.")


EXECUTION TIME SUMMARY
                                Operation   Time (s)  % of Total
                     Build Calliope model 820.445949       62.54
            Create scenario and run model 319.907034       24.39
                     Solve Calliope model 121.265346        9.24
                        Visualize network  24.481679        1.87
               Visualize Calliope results  14.113930        1.08
         Load and filter Stedin grid data   3.417003        0.26
       Clean up electricity grid topology   1.736194        0.13
                 Export bill of materials   1.623867        0.12
              API calls to obtain geodata   1.586876        0.12
     Add links between transmission nodes   0.797642        0.06
             Remove disconnected features   0.761797        0.06
        Neighborhood heat demand plotting   0.647570        0.05
Create and interpolate transmission nodes   0.476394        0.04
             Save DataFrames to CSV files   0.220234        0.02
 